In [1]:
import pandas as pd
import numpy as np
import os

# Load Last.fm scrobbles
print("Loading Last.fm data... (takes 1-2 min)")
chunks = []
users_seen = set()

reader = pd.read_csv(
    "data/userid-timestamp-artid-artname-traid-traname.tsv",
    sep="\t",
    header=None,
    names=["user", "timestamp", "artist_id", "artist", "track_id", "track"],
    usecols=["user", "timestamp", "artist", "track"],
    encoding="utf-8",
    on_bad_lines="skip",
    chunksize=500_000,
)

for chunk in reader:
    chunk = chunk.dropna(subset=["track"])
    users_seen.update(chunk["user"].unique())
    chunks.append(chunk)
    print(f"  Users seen so far: {len(users_seen)}")
    if len(users_seen) >= 500:
        break

df = pd.concat(chunks, ignore_index=True)
top_users = list(df["user"].unique())[:500]
df = df[df["user"].isin(top_users)]

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["timestamp"])
df["timestamp"] = df["timestamp"].astype(np.int64) // 10**9
df = df.sort_values(["user", "timestamp"]).reset_index(drop=True)

print(f"\nDone! {len(df):,} scrobbles from {df['user'].nunique()} users")
print(df.head())

Loading Last.fm data... (takes 1-2 min)
  Users seen so far: 22
  Users seen so far: 41
  Users seen so far: 70
  Users seen so far: 97
  Users seen so far: 129
  Users seen so far: 149
  Users seen so far: 174
  Users seen so far: 203
  Users seen so far: 233
  Users seen so far: 259
  Users seen so far: 280
  Users seen so far: 307
  Users seen so far: 342
  Users seen so far: 357
  Users seen so far: 384
  Users seen so far: 424
  Users seen so far: 447
  Users seen so far: 476
  Users seen so far: 505

Done! 9,409,498 scrobbles from 500 users
          user   timestamp             artist  \
0  user_000001  1155477560  Plaid & Bob Jaroc   
1  user_000001  1155477809  Plaid & Bob Jaroc   
2  user_000001  1155478243  Plaid & Bob Jaroc   
3  user_000001  1155478660     Tommy Guerrero   
4  user_000001  1155478746      Artful Dodger   

                                      track  
0                 The Launching Of Big Face  
1                                   Zn Zero  
2  The Return 

In [2]:
# Load Spotify audio features
print("Loading Spotify audio features...")
sp = pd.read_csv("data/SpotifyAudioFeaturesNov2018.csv")
print(f"Columns: {list(sp.columns)}")
print(f"Shape: {sp.shape}")
print(sp.head(3))

Loading Spotify audio features...
Columns: ['artist_name', 'track_id', 'track_name', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'popularity']
Shape: (116372, 17)
  artist_name                track_id  \
0          YG  2RM4jf1Xa9zPgMGRDiht8O   
1          YG  1tHDG53xJNGsItRA3vfVgs   
2       R3HAB  6Wosx2euFPMT14UXiWudMy   

                                       track_name  acousticness  danceability  \
0  Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj       0.00582         0.743   
1                    BAND DRUM (feat. A$AP Rocky)       0.02440         0.846   
2                                   Radio Silence       0.02500         0.603   

   duration_ms  energy  instrumentalness  key  liveness  loudness  mode  \
0       238373   0.339               0.0    1    0.0812    -7.678     1   
1       214800   0.557               0.0    8    0.2860    -7.259     1 

In [3]:
# Merge scrobbles with Spotify audio features on track name
print("Merging...")

AUDIO_FEATURES = ["valence", "energy", "danceability", "tempo",
                  "acousticness", "instrumentalness", "liveness",
                  "loudness", "speechiness", "mode"]

# normalize track names for matching
df["track_lower"] = df["track"].str.strip().str.lower()
sp["track_lower"] = sp["track_name"].str.strip().str.lower()

# keep only needed spotify columns
sp_slim = sp[["track_lower"] + AUDIO_FEATURES].drop_duplicates(subset=["track_lower"])

# merge
enriched = df.merge(sp_slim, on="track_lower", how="left").drop(columns=["track_lower"])

matched = enriched["valence"].notna().sum()
total   = len(enriched)
print(f"Matched {matched:,} / {total:,} scrobbles ({matched/total:.1%})")

# keep only matched
enriched = enriched.dropna(subset=["valence"]).reset_index(drop=True)

# normalize tempo to 0-1
enriched["tempo"] = (enriched["tempo"] - enriched["tempo"].min()) / (enriched["tempo"].max() - enriched["tempo"].min())

print(f"Final dataset: {len(enriched):,} scrobbles, {enriched['user'].nunique()} users")
print(enriched[["user", "artist", "track", "valence", "energy", "timestamp"]].head())

Merging...
Matched 1,874,238 / 9,409,498 scrobbles (19.9%)
Final dataset: 1,874,238 scrobbles, 499 users
          user        artist    track  valence  energy   timestamp
0  user_000001  Rei Harakami      Joy    0.070   0.352  1155478983
1  user_000001  Rei Harakami     Lust    0.313   0.395  1155480263
2  user_000001        Bayaka  Seasons    0.170   0.359  1155481645
3  user_000001  José Padilla     Solo    0.108   0.545  1155484807
4  user_000001   Chari Chari      One    0.248   0.382  1155644871


In [4]:
# Save enriched data
os.makedirs("data", exist_ok=True)
enriched.to_csv("data/enriched_scrobbles.csv", index=False)
print("Saved → data/enriched_scrobbles.csv")

Saved → data/enriched_scrobbles.csv


In [5]:
# Session segmentation — gap > 30 min = new session
print("Segmenting into sessions...")

SESSION_GAP = 30 * 60  # 30 minutes in seconds
MIN_LENGTH  = 4        # drop sessions shorter than 4 tracks
MOOD_FEATURES = ["valence", "energy", "danceability", "acousticness", "tempo"]

df2 = enriched.sort_values(["user", "timestamp"]).reset_index(drop=True)
df2["time_gap"]    = df2.groupby("user")["timestamp"].diff().fillna(0)
df2["new_session"] = (df2["time_gap"] > SESSION_GAP) | (df2["time_gap"] == 0)
df2["session_id"]  = df2.groupby("user")["new_session"].cumsum()
df2["global_session_id"] = df2["user"] + "_" + df2["session_id"].astype(str)

# filter short sessions
sizes = df2.groupby("global_session_id").size()
valid = sizes[sizes >= MIN_LENGTH].index
df2   = df2[df2["global_session_id"].isin(valid)].reset_index(drop=True)

print(f"Total sessions: {df2['global_session_id'].nunique():,}")
print(f"Total tracks in sessions: {len(df2):,}")
print(f"Avg session length: {df2.groupby('global_session_id').size().mean():.1f} tracks")
df2.to_csv("data/sessions.csv", index=False)
print("Saved → data/sessions.csv")

Segmenting into sessions...
Total sessions: 164,091
Total tracks in sessions: 1,206,762
Avg session length: 7.4 tracks
Saved → data/sessions.csv


In [6]:
# Compute mood arc for each session
print("Computing mood arcs...")

arc_rows = []
grouped = df2.groupby("global_session_id")

for sid, grp in grouped:
    grp = grp.reset_index(drop=True)
    n     = len(grp)
    third = max(1, n // 3)

    start = grp.iloc[:third][MOOD_FEATURES].mean()
    mid   = grp.iloc[third:2*third][MOOD_FEATURES].mean()
    end   = grp.iloc[2*third:][MOOD_FEATURES].mean()

    arc = {"global_session_id": sid, "user": grp["user"].iloc[0]}
    for f in MOOD_FEATURES:
        arc[f"start_{f}"] = start[f]
        arc[f"mid_{f}"]   = mid[f]
        arc[f"end_{f}"]   = end[f]
        arc[f"delta_{f}"] = end[f] - start[f]

    ts = grp["timestamp"].iloc[0]
    dt = pd.to_datetime(ts, unit="s")
    arc["hour_of_day"]    = dt.hour
    arc["day_of_week"]    = dt.dayofweek
    arc["session_length"] = n
    arc["valence_trend"]  = "up" if arc["delta_valence"] > 0.05 else ("down" if arc["delta_valence"] < -0.05 else "flat")
    arc["energy_trend"]   = "up" if arc["delta_energy"]  > 0.05 else ("down" if arc["delta_energy"]  < -0.05 else "flat")
    arc_rows.append(arc)

arcs = pd.DataFrame(arc_rows)
arcs.to_csv("data/session_arcs.csv", index=False)
print(f"Saved {len(arcs):,} session arcs → data/session_arcs.csv")
print(arcs[["global_session_id", "session_length", "valence_trend", "energy_trend"]].head(8))

Computing mood arcs...
Saved 164,091 session arcs → data/session_arcs.csv
  global_session_id  session_length valence_trend energy_trend
0  user_000001_1012               4          down           up
1  user_000001_1018               6          flat           up
2  user_000001_1019               4          flat           up
3  user_000001_1021               9          down         down
4  user_000001_1023               7          down           up
5  user_000001_1024               4          flat         flat
6  user_000001_1028              26          flat         flat
7  user_000001_1038               4          down           up


In [9]:
# Faster clustering using regular k-means on flattened arc features
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import joblib
import os

os.makedirs("models", exist_ok=True)
os.makedirs("plots", exist_ok=True)

N_CLUSTERS = 8
ARC_FEATURES = ["valence", "energy", "danceability", "acousticness"]

print("Building arc sequences...")
arc_cols = [f"{p}_{f}" for p in ["start", "mid", "end"] for f in ARC_FEATURES]
arcs_clean = arcs.dropna(subset=arc_cols).reset_index(drop=True)

# flatten to 2D array instead of DTW (much faster, similar quality)
X = arcs_clean[arc_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering with MiniBatchKMeans (takes ~30 seconds)...")
model = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42,
                        batch_size=10000, max_iter=300, n_init=10)
labels = model.fit_predict(X_scaled)

arcs_clean["trajectory_cluster"] = labels
ARCHETYPE_NAMES = {
    0: "Chill Descent", 1: "Morning Lift", 2: "Late Night Focus",
    3: "Party Arc",     4: "Emotional Journey", 5: "Deep Work",
    6: "Sunset Wind-down", 7: "Euphoric Build",
}
arcs_clean["trajectory_name"] = arcs_clean["trajectory_cluster"].map(ARCHETYPE_NAMES)

arcs_clean.to_csv("data/session_arcs_clustered.csv", index=False)
joblib.dump({"model": model, "scaler": scaler}, "models/trajectory_model.pkl")

print("\nCluster distribution:")
print(arcs_clean["trajectory_name"].value_counts().to_string())
print(f"\nSaved → data/session_arcs_clustered.csv")

Building arc sequences...
Clustering with MiniBatchKMeans (takes ~30 seconds)...

Cluster distribution:
trajectory_name
Emotional Journey    23328
Chill Descent        22700
Sunset Wind-down     21536
Deep Work            21248
Late Night Focus     21204
Party Arc            20628
Morning Lift         20068
Euphoric Build       13379

Saved → data/session_arcs_clustered.csv


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

MOOD_FEATURES = ["valence", "energy", "danceability", "acousticness", "tempo"]
SEED_LENGTH   = 3
HIDDEN_DIM    = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
BATCH_SIZE    = 256
EPOCHS        = 20
LR            = 1e-3

# build sequences
print("Building training sequences...")
label_map = arcs_clean.set_index("global_session_id")["trajectory_cluster"].to_dict()

X_seq, y_seq = [], []
for sid, grp in df2.groupby("global_session_id"):
    if sid not in label_map:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    if len(grp) < SEED_LENGTH:
        continue
    seed = grp.iloc[:SEED_LENGTH][MOOD_FEATURES].values
    if np.isnan(seed).any():
        continue
    X_seq.append(seed)
    y_seq.append(label_map[sid])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.int64)
print(f"Dataset: {len(X_seq):,} samples")

# dataset
class SessionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

X_train, X_val, y_train, y_val = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq)
train_loader = DataLoader(SessionDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SessionDataset(X_val,   y_val),   batch_size=BATCH_SIZE)

# model
class TrajectoryGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(len(MOOD_FEATURES), HIDDEN_DIM, NUM_LAYERS,
                          batch_first=True, dropout=DROPOUT)
        self.fc  = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM // 2, N_CLUSTERS),
        )
    def forward(self, x):
        _, h = self.gru(x)
        return self.fc(h[-1])

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gru_model = TrajectoryGRU().to(device)
optimizer = torch.optim.Adam(gru_model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

print(f"Training on: {device}")
best_val_acc = 0
for epoch in range(1, EPOCHS + 1):
    gru_model.train()
    correct, total = 0, 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(gru_model(Xb), yb)
        loss.backward()
        optimizer.step()
        correct += (gru_model(Xb).argmax(1) == yb).sum().item()
        total   += len(yb)
    tr_acc = correct / total

    gru_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            correct += (gru_model(Xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    val_acc = correct / total
    scheduler.step(1 - val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(gru_model.state_dict(), "models/gru_model.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")

joblib.dump({"input_dim": len(MOOD_FEATURES), "hidden_dim": HIDDEN_DIM,
             "num_layers": NUM_LAYERS, "num_classes": N_CLUSTERS,
             "dropout": DROPOUT, "mood_features": MOOD_FEATURES,
             "seed_length": SEED_LENGTH}, "models/gru_config.pkl")
print(f"\nBest val accuracy: {best_val_acc:.4f}")
print("Saved → models/gru_model.pt")

Building training sequences...
Dataset: 164,091 samples
Training on: cpu
Epoch  1 | train acc 0.404 | val acc 0.461
Epoch  5 | train acc 0.492 | val acc 0.491
Epoch 10 | train acc 0.497 | val acc 0.496
Epoch 15 | train acc 0.499 | val acc 0.499
Epoch 20 | train acc 0.499 | val acc 0.499

Best val accuracy: 0.4995
Saved → models/gru_model.pt


In [11]:
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

print("Building track features...")
track_feats = df2.groupby("track").agg(
    play_count=("user", "count"),
    unique_users=("user", "nunique"),
    mean_valence=("valence", "mean"),
    mean_energy=("energy", "mean"),
    mean_danceability=("danceability", "mean"),
    mean_acousticness=("acousticness", "mean"),
    mean_tempo=("tempo", "mean"),
).reset_index()
track_feats["log_play_count"] = np.log1p(track_feats["play_count"])
track_feats.to_parquet("data/track_features.parquet", index=False)
print(f"Track features: {len(track_feats):,} tracks")

# build training pairs
print("Building training pairs...")
arc_map       = arcs_clean.set_index("global_session_id")
track_feats_idx = track_feats.set_index("track")
top_tracks    = track_feats.nlargest(500, "log_play_count")["track"].tolist()

rows = []
for sid, grp in df2.groupby("global_session_id"):
    if sid not in arc_map.index:
        continue
    arc_row  = arc_map.loc[sid]
    traj_cls = int(arc_row["trajectory_cluster"])
    grp      = grp.sort_values("timestamp").reset_index(drop=True)
    n        = len(grp)
    if n < 6:
        continue

    positives  = set(grp.iloc[n//2:]["track"].tolist())
    seed_set   = set(grp.iloc[:n//2]["track"].tolist())
    candidates = (set(top_tracks) | positives) - seed_set

    for track in list(candidates)[:50]:
        if track not in track_feats_idx.index:
            continue
        tf = track_feats_idx.loc[track]
        rows.append({
            "session_id":         sid,
            "label":              1 if track in positives else 0,
            "trajectory_cluster": traj_cls,
            "hour_of_day":        arc_row["hour_of_day"],
            "day_of_week":        arc_row["day_of_week"],
            "log_play_count":     tf["log_play_count"],
            "unique_users":       tf["unique_users"],
            "track_valence":      tf["mean_valence"],
            "track_energy":       tf["mean_energy"],
            "track_danceability": tf["mean_danceability"],
            "track_acousticness": tf["mean_acousticness"],
            "track_tempo":        tf["mean_tempo"],
        })

pairs = pd.DataFrame(rows)
print(f"Training pairs: {len(pairs):,} | Positive rate: {pairs['label'].mean():.3%}")

feature_cols = [c for c in pairs.columns if c not in ["session_id", "label"]]
X = pairs[feature_cols].fillna(0)
y = pairs["label"]
groups = pairs["session_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

train_data = lgb.Dataset(X.iloc[train_idx], label=y.iloc[train_idx])
val_data   = lgb.Dataset(X.iloc[val_idx],   label=y.iloc[val_idx], reference=train_data)

params = {
    "objective": "binary", "metric": ["auc"],
    "num_leaves": 63, "learning_rate": 0.05,
    "feature_fraction": 0.8, "bagging_fraction": 0.8,
    "bagging_freq": 5, "verbose": -1,
}

print("Training LightGBM ranker...")
model_lgb = lgb.train(
    params, train_data, num_boost_round=300,
    valid_sets=[val_data], valid_names=["val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

joblib.dump({"model": model_lgb, "feature_cols": feature_cols}, "models/lgbm_ranker.pkl")
print(f"\nBest AUC: {model_lgb.best_score['val']['auc']:.4f}")
print("Saved → models/lgbm_ranker.pkl")

Building track features...
Track features: 18,191 tracks
Building training pairs...
Training pairs: 4,267,900 | Positive rate: 0.845%
Training LightGBM ranker...
Training until validation scores don't improve for 30 rounds
[50]	val's auc: 0.899353
Early stopping, best iteration is:
[65]	val's auc: 0.900163

Best AUC: 0.9002
Saved → models/lgbm_ranker.pkl


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

ARCHETYPE_TARGET_MOODS = {
    0: np.array([0.3, 0.2, 0.3, 0.7, 0.3]),
    1: np.array([0.8, 0.8, 0.7, 0.2, 0.7]),
    2: np.array([0.3, 0.4, 0.3, 0.4, 0.4]),
    3: np.array([0.7, 0.9, 0.9, 0.1, 0.8]),
    4: np.array([0.6, 0.5, 0.5, 0.5, 0.5]),
    5: np.array([0.2, 0.3, 0.2, 0.6, 0.3]),
    6: np.array([0.6, 0.3, 0.4, 0.6, 0.3]),
    7: np.array([0.9, 0.9, 0.8, 0.1, 0.9]),
}

lgbm_bundle  = joblib.load("models/lgbm_ranker.pkl")
lgbm_model   = lgbm_bundle["model"]
feature_cols = lgbm_bundle["feature_cols"]
track_feats_idx = track_feats.set_index("track")
arc_map      = arcs_clean.set_index("global_session_id")

EVAL_MOOD_FEATURES = ["mean_valence", "mean_energy", "mean_danceability",
                      "mean_acousticness", "mean_tempo"]

def mood_coherence_score(recommended, target_mood):
    vecs = []
    for track in recommended:
        if track in track_feats_idx.index:
            row = track_feats_idx.loc[track]
            vecs.append([row.get(f, 0.5) for f in EVAL_MOOD_FEATURES])
    if not vecs:
        return 0.0
    sims = cosine_similarity(np.array(vecs), target_mood.reshape(1, -1))
    return float(sims.mean())

def ndcg_at_k(recommended, relevant, k):
    rel = [1 if t in relevant else 0 for t in recommended[:k]]
    ideal = sorted(rel, reverse=True)
    def dcg(r): return sum(v/np.log2(i+2) for i, v in enumerate(r))
    return dcg(rel)/dcg(ideal) if dcg(ideal) > 0 else 0.0

def hit_rate_at_k(recommended, relevant, k):
    return int(any(t in relevant for t in recommended[:k]))

print("Evaluating on 300 sessions...")
results = {"ndcg": [], "hit_rate": [], "mcs_model": [], "mcs_baseline": []}
K = 10
eval_sessions = list(arc_map.index)[:300]
top_popular   = track_feats.nlargest(K, "log_play_count")["track"].tolist()

for sid in eval_sessions:
    grp = df2[df2["global_session_id"] == sid].sort_values("timestamp")
    n   = len(grp)
    if n < 6:
        continue

    relevant   = set(grp.iloc[n//2:]["track"].tolist())
    seed_set   = set(grp.iloc[:n//2]["track"].tolist())
    traj_cls   = int(arc_map.loc[sid]["trajectory_cluster"])
    target     = ARCHETYPE_TARGET_MOODS.get(traj_cls, np.ones(5)*0.5)
    arc_row    = arc_map.loc[sid]

    candidates = track_feats[~track_feats["track"].isin(seed_set)].head(200)
    if len(candidates) == 0:
        continue

    feat_rows = []
    for _, crow in candidates.iterrows():
        feat_rows.append({
            "trajectory_cluster": traj_cls,
            "hour_of_day":        arc_row["hour_of_day"],
            "day_of_week":        arc_row["day_of_week"],
            "log_play_count":     crow["log_play_count"],
            "unique_users":       crow["unique_users"],
            "track_valence":      crow["mean_valence"],
            "track_energy":       crow["mean_energy"],
            "track_danceability": crow["mean_danceability"],
            "track_acousticness": crow["mean_acousticness"],
            "track_tempo":        crow["mean_tempo"],
        })

    X_cand  = pd.DataFrame(feat_rows).reindex(columns=feature_cols, fill_value=0)
    scores  = lgbm_model.predict(X_cand)
    ranked  = [t for _, t in sorted(zip(scores, candidates["track"].tolist()), reverse=True)]

    results["ndcg"].append(ndcg_at_k(ranked, relevant, K))
    results["hit_rate"].append(hit_rate_at_k(ranked, relevant, K))
    results["mcs_model"].append(mood_coherence_score(ranked[:K], target))
    results["mcs_baseline"].append(mood_coherence_score(top_popular, target))

print(f"\n=== Evaluation Results (K={K}) ===")
print(f"NDCG@{K}:               {np.mean(results['ndcg']):.4f}")
print(f"Hit Rate@{K}:           {np.mean(results['hit_rate']):.4f}")
print(f"MCS (our model):       {np.mean(results['mcs_model']):.4f}")
print(f"MCS (popularity base): {np.mean(results['mcs_baseline']):.4f}")
improvement = (np.mean(results['mcs_model']) - np.mean(results['mcs_baseline'])) / np.mean(results['mcs_baseline'])
print(f"MCS improvement:       {improvement:+.1%}")

Evaluating on 300 sessions...

=== Evaluation Results (K=10) ===
NDCG@10:               0.0000
Hit Rate@10:           0.0000
MCS (our model):       0.8640
MCS (popularity base): 0.8497
MCS improvement:       +1.7%


In [13]:
print("Evaluating on 300 sessions...")
results = {"ndcg": [], "hit_rate": [], "mcs_model": [], "mcs_baseline": []}
K = 10
eval_sessions = list(arc_map.index)[:300]
top_popular   = track_feats.nlargest(K, "log_play_count")["track"].tolist()
all_tracks    = set(track_feats["track"].tolist())

for sid in eval_sessions:
    grp = df2[df2["global_session_id"] == sid].sort_values("timestamp")
    n   = len(grp)
    if n < 6:
        continue

    relevant   = set(grp.iloc[n//2:]["track"].tolist()) & all_tracks
    seed_set   = set(grp.iloc[:n//2]["track"].tolist())
    if not relevant:
        continue

    traj_cls   = int(arc_map.loc[sid]["trajectory_cluster"])
    target     = ARCHETYPE_TARGET_MOODS.get(traj_cls, np.ones(5)*0.5)
    arc_row    = arc_map.loc[sid]

    # include relevant tracks in candidates so they can be ranked
    relevant_feats = track_feats[track_feats["track"].isin(relevant)]
    other_feats    = track_feats[~track_feats["track"].isin(seed_set | relevant)].head(200 - len(relevant_feats))
    candidates     = pd.concat([relevant_feats, other_feats]).reset_index(drop=True)

    feat_rows = []
    for _, crow in candidates.iterrows():
        feat_rows.append({
            "trajectory_cluster": traj_cls,
            "hour_of_day":        arc_row["hour_of_day"],
            "day_of_week":        arc_row["day_of_week"],
            "log_play_count":     crow["log_play_count"],
            "unique_users":       crow["unique_users"],
            "track_valence":      crow["mean_valence"],
            "track_energy":       crow["mean_energy"],
            "track_danceability": crow["mean_danceability"],
            "track_acousticness": crow["mean_acousticness"],
            "track_tempo":        crow["mean_tempo"],
        })

    X_cand  = pd.DataFrame(feat_rows).reindex(columns=feature_cols, fill_value=0)
    scores  = lgbm_model.predict(X_cand)
    ranked  = [t for _, t in sorted(zip(scores, candidates["track"].tolist()), reverse=True)]

    results["ndcg"].append(ndcg_at_k(ranked, relevant, K))
    results["hit_rate"].append(hit_rate_at_k(ranked, relevant, K))
    results["mcs_model"].append(mood_coherence_score(ranked[:K], target))
    results["mcs_baseline"].append(mood_coherence_score(top_popular, target))

print(f"\n=== Evaluation Results (K={K}) ===")
print(f"NDCG@{K}:               {np.mean(results['ndcg']):.4f}")
print(f"Hit Rate@{K}:           {np.mean(results['hit_rate']):.4f}")
print(f"MCS (our model):       {np.mean(results['mcs_model']):.4f}")
print(f"MCS (popularity base): {np.mean(results['mcs_baseline']):.4f}")
improvement = (np.mean(results['mcs_model']) - np.mean(results['mcs_baseline'])) / np.mean(results['mcs_baseline'])
print(f"MCS improvement:       {improvement:+.1%}")

Evaluating on 300 sessions...

=== Evaluation Results (K=10) ===
NDCG@10:               0.0307
Hit Rate@10:           0.0786
MCS (our model):       0.8640
MCS (popularity base): 0.8497
MCS improvement:       +1.7%


In [14]:
# save everything the frontend needs
import json

# save track list for frontend
track_list = track_feats["track"].tolist()
with open("data/track_list.json", "w") as f:
    json.dump(track_list, f)

print("All models saved:")
print("  models/gru_model.pt")
print("  models/gru_config.pkl")
print("  models/lgbm_ranker.pkl")
print("  models/trajectory_model.pkl")
print("  data/track_features.parquet")
print("  data/track_list.json")
print("\nReady to launch frontend!")

All models saved:
  models/gru_model.pt
  models/gru_config.pkl
  models/lgbm_ranker.pkl
  models/trajectory_model.pkl
  data/track_features.parquet
  data/track_list.json

Ready to launch frontend!


In [3]:
# Create a dummy ALS model so the API doesn't crash
import implicit
import scipy.sparse as sparse
import joblib

print("Creating ALS model...")
# build a small user-item matrix from our data
plays = df2.groupby(["user", "track"]).size().reset_index(name="plays")
user_idx = {u: i for i, u in enumerate(plays["user"].unique())}
item_idx = {t: i for i, t in enumerate(plays["track"].unique())}

rows = plays["user"].map(user_idx)
cols = plays["track"].map(item_idx)
matrix = sparse.csr_matrix((plays["plays"].values, (rows, cols)),
                            shape=(len(user_idx), len(item_idx)))

als = implicit.als.AlternatingLeastSquares(factors=32, iterations=10, random_state=42)
als.fit(matrix)

joblib.dump(als,      "models/als_model.pkl")
joblib.dump(user_idx, "models/user_idx.pkl")
joblib.dump(item_idx, "models/item_idx.pkl")
print(f"ALS trained. Users: {len(user_idx)}, Items: {len(item_idx)}")
print("Saved → models/als_model.pkl")

Creating ALS model...


C:\Users\bylrs\anaconda3\envs\music-rec\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 22 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/10 [00:00<?, ?it/s]

ALS trained. Users: 493, Items: 18191
Saved → models/als_model.pkl


In [2]:
import pandas as pd
import numpy as np
import joblib

df2 = pd.read_csv("data/sessions.csv")
print(f"Loaded {len(df2):,} session tracks")

Loaded 1,206,762 session tracks


In [4]:
# resave trajectory model with flattened scaler
from sklearn.preprocessing import StandardScaler
import joblib

arc_cols = [f"{p}_{f}" for p in ["start", "mid", "end"] 
            for f in ["valence", "energy", "danceability", "acousticness"]]

arcs_clean = pd.read_csv("data/session_arcs_clustered.csv")
X = arcs_clean[arc_cols].dropna().values

scaler2d = StandardScaler()
scaler2d.fit(X)

bundle = joblib.load("models/trajectory_model.pkl")
bundle["scaler"] = scaler2d
bundle["arc_cols"] = arc_cols
joblib.dump(bundle, "models/trajectory_model.pkl")
print("Resaved trajectory model with 2D scaler")

Resaved trajectory model with 2D scaler


In [6]:
import pandas as pd
track_feats = pd.read_parquet("data/track_features.parquet")
sample_tracks = track_feats.nlargest(20, "play_count")["track"].tolist()
print("Try these tracks in the UI:")
for t in sample_tracks:
    print(f"  - {t}")

Try these tracks in the UI:
  - Intro
  - Heartless
  - Love Lockdown
  - All I Need
  - Say You Will
  - Crazy
  - Street Lights
  - Home
  - One
  - Love Will Tear Us Apart
  - Lullaby
  - Closer
  - Angel
  - Run
  - Untitled
  - Hallelujah
  - Breathe
  - Runaway
  - Hurt
  - Smile


In [7]:
import os

paths = [
    "models/gru_model.pt",
    "models/lgbm_ranker.pkl",
    "models/trajectory_model.pkl",
    "models/als_model.pkl",
    "models/gru_config.pkl",
    "data/track_features.parquet",
    "data/track_list.json",
]

for p in paths:
    size = os.path.getsize(p) / (1024*1024)
    print(f"{p}: {size:.1f} MB")

models/gru_model.pt: 0.6 MB
models/lgbm_ranker.pkl: 0.4 MB
models/trajectory_model.pkl: 0.6 MB
models/als_model.pkl: 2.3 MB
models/gru_config.pkl: 0.0 MB
data/track_features.parquet: 0.6 MB
data/track_list.json: 0.3 MB
